# Error Analysis: Cat vs Dog Classifier

This notebook examines the model's mistakes to understand failure patterns.

In [ ]:
%matplotlib inline

import sys
sys.path.insert(0, '..')

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image
from torch.utils.data import DataLoader

from src.data.dataset import CatDogDataset
from src.data.transforms import inference_transform, IMAGENET_MEAN, IMAGENET_STD
from src.training.evaluate import load_model, get_predictions

## Load Model and Test Data

In [ ]:
model = load_model('../models/best_model.pth')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

test_dataset = CatDogDataset('../data/splits/test', transform=inference_transform)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=4)

labels, preds, probs = get_predictions(model, test_loader, device)
print(f'Test samples: {len(labels)}')
print(f'Accuracy: {(labels == preds).mean():.4f}')

## Find Most Confident Wrong Predictions

These are the images where the model was **most sure** but **wrong** — the most informative errors.

In [ ]:
# Find wrong predictions
wrong_mask = labels != preds
wrong_indices = np.where(wrong_mask)[0]
wrong_confidences = probs[wrong_indices].max(axis=1)

# Sort by confidence (most confident mistakes first)
sorted_order = np.argsort(-wrong_confidences)
top_wrong = wrong_indices[sorted_order[:20]]

print(f'Total wrong predictions: {len(wrong_indices)} / {len(labels)}')
print(f'Error rate: {len(wrong_indices) / len(labels) * 100:.2f}%')

In [ ]:
CLASS_NAMES = ['cat', 'dog']

def denormalize(tensor):
    """Reverse ImageNet normalization for display."""
    mean = np.array(IMAGENET_MEAN)
    std = np.array(IMAGENET_STD)
    img = tensor.numpy().transpose(1, 2, 0)  # CHW -> HWC
    img = img * std + mean
    return np.clip(img, 0, 1)

## Top 20 Most Confident Mistakes

In [ ]:
import os
os.makedirs('../models/evaluation', exist_ok=True)

# Load raw images (without transform) for display
raw_dataset = CatDogDataset('../data/splits/test', transform=None)

n_show = min(20, len(top_wrong))
fig, axes = plt.subplots(4, 5, figsize=(20, 16))
axes = axes.flatten()

for i, idx in enumerate(top_wrong[:n_show]):
    img, _ = raw_dataset[idx]
    true_label = CLASS_NAMES[labels[idx]]
    pred_label = CLASS_NAMES[preds[idx]]
    confidence = probs[idx].max() * 100

    axes[i].imshow(img)
    axes[i].set_title(
        f'True: {true_label}\nPred: {pred_label} ({confidence:.1f}%)',
        color='red', fontsize=10
    )
    axes[i].axis('off')

# Hide unused axes
for i in range(n_show, len(axes)):
    axes[i].axis('off')

plt.suptitle('Top 20 Most Confident Wrong Predictions', fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig('../models/evaluation/top_wrong_predictions.png', dpi=100, bbox_inches='tight')
plt.show()

## Error Pattern Analysis

In [ ]:
# Analyze error distribution by class
wrong_true_labels = labels[wrong_mask]
cat_errors = (wrong_true_labels == 0).sum()
dog_errors = (wrong_true_labels == 1).sum()
total_cats = (labels == 0).sum()
total_dogs = (labels == 1).sum()

print(f'Cat errors: {cat_errors}/{total_cats} ({cat_errors/total_cats*100:.2f}%)')
print(f'Dog errors: {dog_errors}/{total_dogs} ({dog_errors/total_dogs*100:.2f}%)')

In [ ]:
# Confidence distribution: correct vs wrong predictions
correct_mask = labels == preds
correct_conf = probs[correct_mask].max(axis=1)
wrong_conf = probs[wrong_mask].max(axis=1)

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(correct_conf, bins=50, alpha=0.7, label=f'Correct (n={correct_mask.sum()})', color='green')
ax.hist(wrong_conf, bins=50, alpha=0.7, label=f'Wrong (n={wrong_mask.sum()})', color='red')
ax.set_xlabel('Confidence')
ax.set_ylabel('Count')
ax.set_title('Confidence Distribution: Correct vs Wrong Predictions')
ax.legend()
plt.savefig('../models/evaluation/confidence_distribution.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# Image size analysis of wrong predictions
wrong_sizes = []
for idx in wrong_indices:
    img, _ = raw_dataset[idx]
    wrong_sizes.append(img.size)  # (width, height)

wrong_widths = [s[0] for s in wrong_sizes]
wrong_heights = [s[1] for s in wrong_sizes]

print(f'Wrong prediction image sizes:')
print(f'  Width  — min: {min(wrong_widths)}, max: {max(wrong_widths)}, mean: {np.mean(wrong_widths):.0f}')
print(f'  Height — min: {min(wrong_heights)}, max: {max(wrong_heights)}, mean: {np.mean(wrong_heights):.0f}')

## Summary

**Key findings from error analysis:**

1. Review the top wrong predictions above to identify visual patterns
2. Check if errors are biased toward one class
3. Check if low-resolution or unusual images cause more errors
4. The confidence distribution shows whether the model is well-calibrated

These findings are documented in `MODEL_CARD.md`.